# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name', metadata)}: {getattr(metadata, 'description', metadata)}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets with their @id
record_sets = list(dataset.record_sets())
print('Available Record Sets:')
for rs in record_sets:
    print(f"@id: {getattr(rs, '@id', None)}, Name: {getattr(rs, 'name', None)}")

# Show fields for each record set
for rs in record_sets:
    print(f"\nFields for Record Set '@id': {getattr(rs, '@id', None)} (Name: {getattr(rs, 'name', None)}):")
    for field in getattr(rs, 'fields', []):
        print(f"  Field @id: {getattr(field, '@id', None)}, Name: {getattr(field, 'name', None)}, DataType: {getattr(field, 'data_type', None)}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Gather @id for each record set
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets]

dataframes = {}
# Load records from each record set into a DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Display the fields (columns) of the main protein record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in record set '@id': {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No records found for the main record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

# Identify a numeric field in the first record set for analysis
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    # Heuristically select a likely numeric field
    numeric_candidates = [col for col in df.columns if any(s in col.lower() for s in ["count", "number", "mw", "abundance", "score", "coverage", "intensity"])]
    numeric_field = numeric_candidates[0] if numeric_candidates else df.select_dtypes(include=np.number).columns[0] if not df.select_dtypes(include=np.number).empty else None
    print(f"Selected numeric field for analysis: {numeric_field}")
    
    # Try to convert field to numeric if not already
    if numeric_field and not np.issubdtype(df[numeric_field].dtype, np.number):
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

    threshold = df[numeric_field].mean() if numeric_field else 10
    filtered_df = df[df[numeric_field] > threshold] if numeric_field else df

    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    if numeric_field:
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Try grouping by a likely categorical field
        group_candidates = [col for col in df.columns if any(s in col.lower() for s in ["accession", "sample", "category", "description", "protein"])]
        group_field = None
        for candidate in group_candidates:
            if candidate != numeric_field and df[candidate].nunique() < df.shape[0] // 2:
                group_field = candidate
                break

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
else:
    print("No suitable numeric field found or no dataframe loaded.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field:
    plt.figure(figsize=(7,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), bins=30, kde=True, color='steelblue')
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    # Scatter plot versus group_field (if present)
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id])
        plt.title(f'{numeric_field} by {group_field}')
        plt.xticks(rotation=90)
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded the FAIR^2 dataset using its Croissant schema.
- Explored available record sets and fields, referencing all entities by their `@id` as required by the schema.
- Examined the distribution of a key numeric attribute (automatically selected) and demonstrated filtering, normalization, and grouping.
- Displayed visualizations to show data trends and grouping effects.

This notebook can be further extended for deeper analysis or integration with other ML workflows. For more information, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant) or inspect the schema directly.